In [164]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [165]:
df_customer=pd.read_csv(r"D:/olist_ecommerce_data/data/olist_customers_dataset.csv")
df_orders=pd.read_csv(r"D:/olist_ecommerce_data/data/olist_orders_dataset.csv")
df_order_items=pd.read_csv(r"D:/olist_ecommerce_data/data/olist_order_items_dataset.csv")
df_products=pd.read_csv(r"D:/olist_ecommerce_data/data/olist_products_dataset.csv")
df_geolocation=pd.read_csv(r"D:/olist_ecommerce_data/data/olist_geolocation_dataset.csv")
df_order_payment=pd.read_csv(r"D:/olist_ecommerce_data/data/olist_order_payments_dataset.csv")
df_seller=pd.read_csv(r"D:/olist_ecommerce_data/data/olist_sellers_dataset.csv")
df_prd_name_trans=pd.read_csv(r"D:/olist_ecommerce_data/data/product_category_name_translation.csv")
df_review=pd.read_csv(r"D:/olist_ecommerce_data/data/olist_order_reviews_dataset.csv")
df_state_full_name=pd.read_csv(r"D:/olist_ecommerce_data/data/state_full_name.csv")

# Visualization Functions 

In [166]:
def pie_plot(data,labels,color_list,title):
    plt.figure(figsize=(8, 8))
    plt.pie(data,labels=labels,autopct='%1.1f%',color=color_list,startangle=90,shadow=True)
    plt.title(title,fontsize=16,fontweight="bold")
    plt.axis("equal")
    plt.tight_layout()
    plt.show()

In [167]:
def histogram(data,column,title,x_label,bins,kde):
    plt.figure(figsize=(10,6))
    sns.histplot(data=data[column],bins=bins,kde=kde)
    plt.title(title,fontsize=13)
    plt.xlabel(x_label,fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
def line_plot(data,x_axis,y_axis,ylabel,title,color,marker):
    plt.figure(figsize=(10,5))
    sns.lineplot(data=data,x=x_axis,y=y_axis,color=color,marker='o',linewidth=1.5)
    plt.title(title,fontsize=14,fontweight="bold")
    plt.ylabel(ylabel,fontsize=14)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
def bar_plot(data,x_axis,y_axis,title,xlabel,ylabel,palette):
    plt.figure(figsize=(10,5))
    sns.barplot(data=data,x=x_axis,y=y_axis,palette=palette,errorbar=None)
    plt.title(title,fontsize=14,fontweight="bold")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Dataset Overview and cleaning

In [ ]:
df_customer.info()

In [ ]:
df_customer.isnull().sum()

In [ ]:
df_orders.info()

In [ ]:
df_orders.isnull().sum()

In [ ]:
df_order_items.isnull().sum()

In [ ]:
df_products.info()

In [ ]:
df_orders.columns

In [ ]:
col=['order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date']
for i in col:
    df_orders[i]=pd.to_datetime(df_orders[i])

In [ ]:
df_orders['order_purchase_timestamp']

In [ ]:
df_orders.head()

In [ ]:
df_products.head()

In [ ]:
for i in df_products.columns:
    print(f"{i :30s} : {df_products[i].isnull().sum()}")

In [ ]:
df_products["product_category_name"].fillna("Unknown",inplace=True)

In [ ]:
for i in df_products.select_dtypes(include="number").columns:
    df_products[i].fillna(df_products[i].median(),inplace=True)

In [ ]:
df_order_payment.info()

In [ ]:
df_seller.info()

In [ ]:
df_prd_name_trans.info()

In [ ]:
df_review.info()

In [ ]:
df_review.head()

# Create Master Dataset

In [ ]:
master_df=df_customer.merge(df_orders,on="customer_id",how="left")
master_df=master_df.merge(df_order_items,on="order_id",how="left")
master_df=master_df.merge(df_products,on="product_id",how="left")
master_df=master_df.merge(df_prd_name_trans,on="product_category_name",how='left')
master_df=master_df.merge(df_seller,on="seller_id",how="left")
master_df=master_df.merge(df_order_payment,on="order_id",how="left")
master_df=master_df.merge(df_review,on="order_id",how="left")


In [ ]:
master_df

In [ ]:
for i in master_df.columns:
    print(f"{i :22s} : {master_df[i].isnull().sum()}")

In [ ]:
master_df.select_dtypes(include="object").columns

In [ ]:
d=['shipping_limit_date','review_creation_date','review_answer_timestamp']
for i in d:
    master_df[i]=pd.to_datetime(master_df[i])

In [ ]:
master_df.columns

# Bussiness Overview

In [ ]:
#Revenue
total_revenue=master_df["payment_value"].sum()
total_orders=master_df["order_id"].nunique()
total_customers=master_df["customer_unique_id"].count()
total_sellers=master_df["seller_id"].nunique()

average_order_value = (
    master_df.groupby("order_id")["payment_value"]
             .max()
             .mean()
)

average_review = master_df["review_score"].mean()

delivered_orders = (
    master_df["order_status"]
    .eq("delivered")
    .mean()*100
)

cancelled_orders = (
    master_df["order_status"]
    .eq("canceled")
    .mean()*100
)

In [ ]:
print(f"Total Revenue    :      {total_revenue :,.2f} BRL")
print(f"Total Orders     :      {total_orders :,}")
print(f"Total Customers  :      {total_customers :,}")
print(f"Total Sellers    :      {total_sellers :,}")
print(f"AOV              :      {average_order_value :.2f} BRL")
print(f"Average Rating   :      {average_review :.2f}")
print(f"Delivered Orders :      {delivered_orders :.2f}%")
print(f"Cancelled Orders :      {cancelled_orders :.2f}%")

# Analysis

## Sales Analysis

In [ ]:
master_df["order_purchase_timestamp"].dt.to_period('M').astype("str")

In [ ]:
sales_df=df_orders.merge(df_order_payment.groupby("order_id",as_index=False)["payment_value"].sum(),on="order_id",how="left")

In [ ]:
sales_df["year_month"]=sales_df["order_purchase_timestamp"].dt.to_period("M").astype("str")

In [ ]:
monthly_sales=sales_df.groupby("year_month")["payment_value"].sum().reset_index()

In [ ]:
monthly_sales.describe()

In [ ]:
monthly_sales.sort_values(by="payment_value",ascending=False).head()

In [ ]:
lowest_sale_month=monthly_sales.sort_values(by="payment_value",ascending=True).head()
lowest_sale_month

In [ ]:
monthly_sales["payment_value"].sum()

In [ ]:
monthly_sales["payment_value"].mean()

In [ ]:
print(f"Average Monthly Sales : {monthly_sales["payment_value"].mean():,.2f} BRL")
print(f"Highest Monthly Sales : {monthly_sales["payment_value"].max() :,.2f} BRL")
print(f"Lowest  Monthly Sales : {monthly_sales["payment_value"].min() :,.2f} BRL")

* On average, Company generated about 640 thousands BRL in revenue each month.
* November 2017 had recorded highest monthly revenue.
* December 2018 had lowest monthly revenue as there were very few orders recorded this month.

In [ ]:
line_plot(monthly_sales,"year_month","payment_value","Revenue","Monthly Sales Trend","#1565C0","o")

* The average monthly revenue is 640,354.88 BRL
* November 2017 had recorded highest monthly revenue(1,194,882.80 BRL).
* September 2016 and the last months of 2018 show very low revenue because the data for these months is incomplete.
* Overall, revenue increased steadily during 2017 and remained high throughout most of 2018.

#### Q2:What is the average order value

In [ ]:
order_value=sales_df.groupby("order_id")["payment_value"].sum().reset_index()

In [ ]:
order_value["payment_value"].describe()

In [ ]:
top_orders = order_value.sort_values(by="payment_value",ascending=False).head()
top_orders

In [ ]:
lowest_order_value=order_value.sort_values(by="payment_value",ascending=True).head()
lowest_order_value

In [ ]:
print(f"Average Order Value: {order_value['payment_value'].mean():,.2f} BRL")
print(f"Highest Order Value: {order_value['payment_value'].max():,.2f} BRL")
print(f"Lowest Order Value: {order_value['payment_value'].min():,.2f} BRL")

In [ ]:
histogram(
    data=order_value,
    column="payment_value",
    title="Figure 5. Distribution of Order Values",
    x_label="Order Value (BRL)",bins=50,kde=False
)

* The average order value is 160.99 BRL.
* Most orders have relatively low values.
* A few orders have very high values, reaching up to 13,664.08 BRL.

Q3:Which days of the week generate the highest revenue?

In [ ]:
sales_df["weekday"] = sales_df["order_purchase_timestamp"].dt.day_name()
weekday_revenue=sales_df.groupby("weekday")["payment_value"].sum().reindex(["Monday","Tuesday","Wednesday",
                                                            "Thursday","Friday","Saturday","Sunday"]).reset_index()
weekday_revenue

In [ ]:
weekday_revenue.sort_values(
    by="payment_value",
    ascending=False
)

In [ ]:
print(f"Highest Revenue: {weekday_revenue['payment_value'].max():,.2f} BRL")
print(f"Lowest Revenue: {weekday_revenue['payment_value'].min():,.2f} BRL")

In [ ]:
bar_plot(
    data=weekday_revenue,
    x_axis="weekday",
    y_axis="payment_value",
    title="Revenue by Weekday",
    xlabel="Weekday",
    ylabel="Revenue (BRL)",palette="deep"
)

* Monday generated the highest revenue (2,622,457.97 BRL).
* Saturday generated the lowest revenue (1,768,427.68 BRL).
* Revenue is generally higher on weekdays than weekends.

### Order Analysis

In [ ]:
monthly_orders=sales_df.groupby("year_month")["order_id"].count().reset_index(name="total_orders")
monthly_orders.head()

In [ ]:
monthly_orders.describe()

In [ ]:
top_orders_month=monthly_orders.sort_values(by="total_orders",ascending=False).head(5)
top_orders_month

In [ ]:
lowest_orders_month=monthly_orders.sort_values(by="total_orders",ascending=True).head(5)
lowest_orders_month

In [ ]:
print(f"Average Monthly Orders : {monthly_orders['total_orders'].mean():,.0f}")
print(f"Highest Monthly Orders : {monthly_orders['total_orders'].max():,.0f}")
print(f"Lowest Monthly Orders  : {monthly_orders['total_orders'].min():,.0f}")

In [ ]:
line_plot(data=monthly_orders,x_axis="year_month",y_axis="total_orders",title="Monthly Order Trend",color="blue",ylabel="Orders",marker="o")

* The average number of orders per month is 3,978.
* November 2017 recorded the highest number of orders (7,544 orders).
* September 2016, December 2016, and the last months of 2018 recorded the lowest number of orders due to incomplete data.
* The number of orders increased steadily during 2017 and remained relatively stable throughout most of 2018.

* Q3:Which states generate the highest revenue?

In [ ]:
state_revenue = (
    master_df.groupby("customer_state")["payment_value"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

state_revenue.head()

In [ ]:
state_revenue=df_state_full_name.merge(state_revenue,on="customer_state",how="left").sort_values(by="payment_value",ascending=False)

In [ ]:
state_revenue.describe()

In [ ]:
state_revenue.head(10)

In [ ]:
print(f"Number of States: {state_revenue.shape[0]}")
print(f"Highest Revenue: {state_revenue['payment_value'].max():,.2f} BRL")
print(f"Lowest Revenue: {state_revenue['payment_value'].min():,.2f} BRL")

In [ ]:
top10_states = state_revenue.head(10)
top10_states


In [ ]:
bar_plot(data=top10_states,x_axis="payment_value",y_axis="State",title="Top 10 State Wise Revenue",xlabel="Revenue",ylabel="States",palette="Set2")

* The dataset includes customers from 27 Brazilian states.
* São Paulo (SP) generated the highest revenue with 7.73 million BRL.
* Rio de Janeiro (RJ) and Minas Gerais (MG) ranked second and third in total revenue.
* A few states contribute a large share of the total revenue, while several states generate much lower sales.

* #### Q4:Which states have the highest number of customers?

In [ ]:
master_df.head()

In [ ]:
state_customers = (
    df_customer.groupby("customer_state")["customer_unique_id"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="total_customers")
)

state_customers.head()

In [ ]:
state_customers = (
    df_customer.groupby("customer_state")["customer_unique_id"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="total_customers")
)

state_customers

In [ ]:
state_customers["total_customers"].describe()

In [ ]:
top10_customers = (
    state_customers.sort_values(
        by="total_customers",
        ascending=False
    )
    .head(10)
)

top10_customers

In [ ]:
lowest_customer_states = (
    state_customers.sort_values(
        by="total_customers",
        ascending=True
    )
    .head(5)
)

lowest_customer_states

In [ ]:
print(f"Number of States: {state_customers.shape[0]}")
print(f"Average Customers per State: {state_customers['total_customers'].mean():,.0f}")
print(f"Highest Number of Customers: {state_customers['total_customers'].max():,.0f}")
print(f"Lowest Number of Customers: {state_customers['total_customers'].min():,.0f}")

In [ ]:
bar_plot(data=top10_customers,
    x_axis="total_customers",
    y_axis="customer_state",
    title="Top 10 States by Number of Customers",
    xlabel="Number of Customers",
    ylabel="State",palette="Set3")

* The dataset includes customers from 27 Brazilian states.
* São Paulo (SP) has the largest customer base with 40,302 customers.
* Rio de Janeiro (RJ) and Minas Gerais (MG) rank second and third in the number of customers.
* States such as Roraima (RR), Amapá (AP), and Acre (AC) have the fewest customers.

## Product Analysis

* Q1:Which product categories generate the highest revenue?

In [ ]:
category_revenue = (
    master_df.groupby("product_category_name_english")["price"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

category_revenue.head()

In [ ]:
category_revenue["Revenue %"] = (category_revenue["price"]/ category_revenue["price"].sum()) * 100
category_revenue.head(10)

In [ ]:
top10_categories = category_revenue.head(10)

top10_share = top10_categories["Revenue %"].sum()

print(f"Top 10 Categories Contribution: {top10_share:.2f}%")

In [ ]:
bar_plot(data=top10_categories,
    x_axis="price",
    y_axis="product_category_name_english",
    title="Top 10 Product Categories by Revenue",
    xlabel="Revenue (BRL)",
    ylabel="Category",palette="Set3")

* Health & Beauty generated the highest revenue (1,301,947.97 BRL), followed by Watches & Gifts and Bed, Bath & Table.
* The revenue among the top five categories is relatively close, indicating strong demand across multiple product categories rather than relying on a single category.
* The top 10 product categories contribute 63.13% of the total revenue, showing that a relatively small number of categories drive most of the company's sales.

* Q2:Which product categories receive the highest number of orders?

In [ ]:
category_orders = (
    master_df.groupby("product_category_name_english")["order_id"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="Total Orders")
)

category_orders.head(10)

In [ ]:
category_orders["Orders %"] = (
    category_orders["Total Orders"]
    / category_orders["Total Orders"].sum()
) * 100

category_orders.head(10)

In [ ]:
top10_orders = category_orders.head(10)

top10_orders_share = top10_orders["Orders %"].sum()

print(f"Top 10 Categories Contribution: {top10_orders_share:.2f}%")

* Bed, Bath & Table received the highest number of orders (9,417 orders), followed by Health & Beauty and Sports & Leisure.
* Although Health & Beauty generated the highest revenue, Bed, Bath & Table received more orders, suggesting that its products may have lower average prices.
* The top 10 product categories account for 62.45% of all orders, indicating that customer demand is concentrated in a limited number of product categories.

In [ ]:
master_df["price"].describe()

In [ ]:
print(f"Mean Price: {master_df['price'].mean():.2f} BRL")
print(f"Median Price: {master_df['price'].median():.2f} BRL")
print(f"Maximum Price: {master_df['price'].max():.2f} BRL")
print(f"Minimum Price: {master_df['price'].min():.2f} BRL")
print(f"Skewness: {master_df['price'].skew():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

price_limit = master_df["price"].quantile(0.99)

ax.hist(
    master_df.loc[master_df["price"] <= price_limit, "price"],
    bins=30,
    color="blue",
    edgecolor="white"
)

ax.set_title("Distribution of Product Prices")
ax.set_xlabel("Product Price (BRL)")
ax.set_ylabel("Frequency")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

* The average product price is 120.65 BRL, while the median price is 74.90 BRL.
* Since the mean is much higher than the median, most products are relatively low-priced, with a small number of very expensive products.
* The price distribution is highly right-skewed (Skewness = 7.89), indicating the presence of high-priced outliers.

In [ ]:
price_freight = master_df[["price", "freight_value"]].dropna()

price_freight.head()

In [ ]:
correlation = price_freight["price"].corr(price_freight["freight_value"])

print(f"Correlation: {correlation:.2f}")

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=master_df.sample(5000,random_state=42),x="price",y="freight_value")
plt.title("Product Price Vs Freight Value",fontweight="bold",fontsize="15")
plt.xlabel("Product Price (BRL)",fontsize="12")
plt.ylabel("Freight Value",fontsize="12")
plt.tight_layout()
plt.show()

* There is a moderate positive correlation (0.42) between product price and freight cost.
* In general, higher-priced products tend to have higher shipping costs.
* However, the relationship is not strong, indicating that freight cost is also influenced by other factors such as product size, weight, and delivery distance.

## Delivery Analysis

* Q1:How long does delivery take?

In [ ]:
master_df["delivery_days"] = (
    master_df["order_delivered_customer_date"] -
    master_df["order_purchase_timestamp"]
).dt.days
master_df["delivery_days"]

In [ ]:
delivery_df = master_df[
    master_df["delivery_days"].notna()
].copy()

In [ ]:
delivery_df["delivery_days"].describe()

In [ ]:
print(f"Average Delivery Time :   {delivery_df['delivery_days'].mean():.1f} days")
print(f"Median Delivery Time  :   {delivery_df['delivery_days'].median():.0f} days")
print(f"Fastest Delivery      :   {delivery_df['delivery_days'].min():.0f} days")
print(f"Longest Delivery      :   {delivery_df['delivery_days'].max():.0f} days")
print(f"Skewness              :   {delivery_df['delivery_days'].skew():.2f}")

In [ ]:
delivery_limit = delivery_df["delivery_days"].quantile(0.99)

histogram(data=delivery_df[delivery_df["delivery_days"] <= delivery_limit]
          ,column="delivery_days",title="Distribution of Delivery Time",x_label="Delivery Time (Days)",bins=40,kde=False)


* The average delivery time is 12 days, while half of the delivered orders arrived within 10 days or less.
* Most orders were delivered within 6 to 15 days.
* The delivery time distribution is positively skewed (Skewness = 3.85), indicating that while most deliveries are completed within a reasonable time, a small number of orders experienced significantly longer delivery times, reaching up to 209 days.

In [ ]:
master_df.select_dtypes(include="datetime").columns

In [ ]:
on_time_df=master_df[master_df["order_delivered_customer_date"].notna()].copy()

In [ ]:
# 'order_delivered_customer_date',
#        'order_estimated_delivery_date'
def delivery_status(order_delivered_customer_date,order_estimated_delivery_date):
    if order_delivered_customer_date<=order_estimated_delivery_date:
        return "On Time"
    else:
        return "Delayed"
        

In [ ]:
on_time_df["delivery_status"]=on_time_df.apply(lambda row:
                                               delivery_status(row["order_delivered_customer_date"],row["order_estimated_delivery_date"]),axis=1)

In [ ]:
on_time_df.shape

In [ ]:
delivery_status=on_time_df["delivery_status"].value_counts().reset_index()

In [ ]:
delivery_status.columns=["delivery_status","Orders"]

In [ ]:
delivery_status["Percentage"]=(delivery_status["Orders"]/delivery_status["Orders"].sum())*100
delivery_status

In [ ]:
bar_plot(data=delivery_status,x_axis="delivery_status",y_axis="Orders",
         xlabel="Delivery Status",ylabel="Orders",title="On Time vs Delayed Deliveries",palette="viridis")

* About 92.16% of orders were delivered on or before the estimated delivery date.
* Only 7.84% of orders were delayed beyond the estimated delivery date.
* This high on-time delivery rate indicates that the company's logistics and delivery operations perform efficiently for most orders.

## Customer Reviews

* Q1:How are customer review scores distributed?

In [ ]:
review_distribution = (
    master_df.groupby("review_score")
    .size()
    .reset_index(name="Number of Reviews")
)

review_distribution["Percentage"] = (
    review_distribution["Number of Reviews"]
    / review_distribution["Number of Reviews"].sum()
) * 100

review_distribution

In [ ]:
bar_plot(
    data=review_distribution,
    x_axis="review_score",
    y_axis="Number of Reviews",
    title="Distribution of Customer Review Scores",
    xlabel="Review Score",
    ylabel="Number of Reviews",palette="viridis"
)

* More than half of customers (56.15%) gave the highest rating (5 stars), indicating a high level of customer satisfaction.
* Low ratings (1 and 2 stars) account for only 16.58% of all reviews, while 4- and 5-star ratings together represent 75.04% of the reviews.
* Overall, customer feedback is largely positive, suggesting that most customers had a satisfactory shopping experience.

Q2:Does delivery time affect customer satisfaction?

In [ ]:
review_delivery =master_df.groupby("review_score")["delivery_days"].mean().reset_index()

review_delivery

In [ ]:
bar_plot(
    data=review_delivery,
    x_axis="review_score",
    y_axis="delivery_days",
    title="Average Delivery Time by Review Score",
    xlabel="Review Score",
    ylabel="Average Delivery Time (Days)",
    palette="rainbow"
)

In [ ]:
correlation = master_df["delivery_days"].corr(master_df["review_score"])
print(f"Correlation: {correlation:.2f}")

* Customers who gave 5-star ratings received their orders in an average of 10.20 days, while customers who gave 1-star ratings waited an average of 19.10 days.
* The correlation between delivery time and review score is -0.30, indicating a moderate negative relationship.
* This suggests that longer delivery times are generally associated with lower customer satisfaction, although other factors also influence customer reviews.

* Q3:Which order status receives the highest ratings

In [ ]:
status_reviews = (
    master_df.groupby("order_status")["review_score"]
    .mean()
    .round(2)
    .sort_values(ascending=False)
    .reset_index()
)
status_reviews

In [ ]:
status_review_summary = (
    master_df.groupby("order_status")
    .agg(
        Average_Review=("review_score", "mean"),
        Number_of_Reviews=("review_score", "count")
    )
    .reset_index()
)

status_review_summary["Average_Review"] = (
    status_review_summary["Average_Review"].round(2)
)

status_review_summary.sort_values(
    by="Average_Review",
    ascending=False
)

In [ ]:
bar_plot(
    data=status_review_summary,
    x_axis="order_status",
    y_axis="Average_Review",
    title="Figure 15. Average Review Score by Order Status",
    xlabel="Order Status",
    ylabel="Average Review Score",
    palette="viridis"
)

* Delivered orders received the highest average review score (4.08) based on 114,862 customer reviews, indicating a high level of customer satisfaction after successful order completion.
* The remaining order statuses received average ratings below 2.0, reflecting low customer satisfaction for orders that were canceled, unavailable, delayed, or still in progress.
* Although the created and approved statuses have slightly higher average ratings than some other non-delivered statuses, they are based on only 3 reviews each and are therefore not statistically representative.

## Payment Analysis

* Q1:Which payment methods are most commonly used?Q1:

In [ ]:
payment_methods = (
    master_df.groupby("payment_type")["order_id"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="Number of Orders")
)

payment_methods["Percentage"] = (
    payment_methods["Number of Orders"]
    / payment_methods["Number of Orders"].sum()
) * 100

payment_methods

In [ ]:
bar_plot(
    data=payment_methods,
    x_axis="payment_type",
    y_axis="Number of Orders",
    title="Payment Methods Used by Customers",
    xlabel="Payment Method",
    ylabel="Number of Orders",
    palette="pastel"
)

* Credit cards are the most frequently used payment method, accounting for 75.24% of all orders.
* Boleto is the second most popular payment method with 19.46%, while voucher and debit card are used much less frequently.
* The results indicate that customers strongly prefer electronic card payments over other payment options.

* Q2:Which payment methods generate the highest revenue?

In [ ]:
payment_revenue = (
    master_df.groupby("payment_type")["payment_value"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

payment_revenue["Revenue %"] = (
    payment_revenue["payment_value"]
    / payment_revenue["payment_value"].sum()
) * 100

payment_revenue

In [ ]:
bar_plot(
    data=payment_revenue,
    x_axis="payment_type",
    y_axis="payment_value",
    title="Revenue by Payment Method",
    xlabel="Payment Method",
    ylabel="Revenue (BRL)",
    palette="icefire"
)

* Credit card payments generated 76.66% of the total revenue, making it the company's primary source of sales.
* Boleto contributed 19.98% of total revenue, while voucher and debit card together represented less than 4%.
* The revenue distribution closely follows the popularity of payment methods, confirming customers' strong preference for credit card payments.

* Q3:How many installments do customers usually choose?

In [ ]:
installments = (
    master_df.groupby("payment_installments")
    .size()
    .reset_index(name="Orders"))

installments["Percentage"] = (
    installments["Orders"]
    / installments["Orders"].sum()) * 100

installments["payment_installments"]=installments["payment_installments"].astype("int")
installments.sort_values(by="Orders",ascending=False).head(10)

In [ ]:
bar_plot(
    data=installments.sort_values(by="payment_installments"),
    x_axis="payment_installments",
    y_axis="Orders",
    title="Number of Payment Installments",
    xlabel="Installments",
    ylabel="Orders",
    palette="icefire",
   
)

* Nearly half of all purchases (49.90%) were paid in a single installment.
* Two and three installments are also common, accounting for 11.61% and 9.98% of orders, respectively.
* The use of installment payments gradually decreases as the number of installments increases, indicating that customers generally prefer shorter payment plans.

## Final Finding

* The business generated approximately **20.58 million BRL in total revenue** from **99,441 orders**.
* Sales showed a consistent upward trend from **2017**, reaching their highest levels in **late 2017 and early 2018**.
* **São Paulo (SP)** was the top-performing state, contributing the highest revenue and having the largest customer base.
* **Health & Beauty, Watches & Gifts, and Bed & Bath Table** were the leading product categories in terms of revenue.
* Product prices were generally concentrated in the lower range, while a small number of high-priced products resulted in a **right-skewed price distribution**.
* The **average delivery time was 12 days**, with approximately **92.16% of orders delivered on or before the estimated delivery date**.
* Overall customer satisfaction was strong, with **56.15% of reviews receiving a 5-star rating**.
* Delivery time had a **moderate negative correlation (-0.30)** with review scores, indicating that **shorter delivery times tend to be associated with higher customer satisfaction**.
* **Credit cards** were the most widely used payment method, representing around **75% of all orders** and contributing more than **76% of total revenue**.
